In [1]:
import os
import sys
from glob import glob
import numpy as np
import dask
import xarray as xr
import xgcm
from xgcm.autogenerate import generate_grid_ds
from cmocean import cm
# import xscale as xsc
import xroms
import gc
from netCDF4 import Dataset

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
%matplotlib inline
import seawater as sw
# from pyspec import spectrum as spec

import time
from tqdm import tqdm

os.chdir('/meddy/simingzhang/Analysis/python/python3/SF3_RLS')
from strucFunct import *
from spectralanalysis import *
from structfunc2KEflux import *

/tmp/ipykernel_1911766/1948973400.py:18: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


In [2]:
def u2rho_2d_xrray(var_u):
    [Mp, L] = var_u.shape
    Lp = L + 1
    Lm = L - 1
    
    # 使用 xarray 创建新变量
    var_rho = xr.DataArray(np.zeros((Mp, Lp)), coords=var_u.coords, dims=var_u.dims)
    
    # 计算 var_rho
    var_rho[:, 1:L-1] = 0.5 * (var_u[:, 0:Lm-1] + var_u[:, 1:L-1])
    
    # 边界条件
    var_rho[:, 0] = var_rho[:, 1]  # 左边界
    var_rho[:, Lp-1] = var_rho[:, -2]  # 右边界
    
    return var_rho

def u2rho_3d_xrray(var_u):
    [N, Mp, L] = var_u.shape
    Lp = L + 1
    Lm = L - 1
    
    # 使用 xarray 创建新变量
    var_rho = xr.DataArray(np.zeros((N, Mp, Lp)), coords=var_u.coords, dims=var_u.dims)
    
    # 计算 var_rho
    var_rho[:, :, 1:L-1] = 0.5 * (var_u[:, :, 0:Lm-1] + var_u[:, :, 1:L-1])
    
    # 边界条件
    var_rho[:, :, 0] = var_rho[:, :, 1]  # 左边界
    var_rho[:, :, Lp-1] = var_rho[:, :, -2]  # 右边界
    
    return var_rho

def v2rho_2d_xrray(var_v):
    [M, Lp] = var_v.shape
    Mp = M + 1
    Mm = M - 1
    
    # 使用 xarray 创建新变量
    var_rho = xr.DataArray(np.zeros((Mp, Lp)), coords=var_v.coords, dims=var_v.dims)
    
    # 计算 var_rho
    var_rho[1:M-1, :] = 0.5 * (var_v[0:Mm-1, :] + var_v[1:M-1, :])
    
    # 边界条件
    var_rho[0, :] = var_rho[1, :]  # 上边界
    var_rho[Mp-1, :] = var_rho[-2, :]  # 下边界
    
    return var_rho

def v2rho_3d_xrray(var_v):
    [N, M, Lp] = var_v.shape
    Mp = M + 1
    Mm = M - 1
    
    # 使用 xarray 创建新变量
    var_rho = xr.DataArray(np.zeros((N, Mp, Lp)), coords=var_v.coords, dims=var_v.dims)
    
    # 计算 var_rho
    var_rho[:, 1:M-1, :] = 0.5 * (var_v[:, 0:Mm-1, :] + var_v[:, 1:M-1, :])
    
    # 边界条件
    var_rho[:, 0, :] = var_rho[:, 1, :]  # 上边界
    var_rho[:, Mp-1, :] = var_rho[:, -2, :]  # 下边界
    
    return var_rho

def u2rho_2d (var_u):
    [Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[:,1:L-1]=0.5*(var_u[:,0:Lm-1]+var_u[:,1:L-1])
    var_rho[:,0]=var_rho[:,1]
    var_rho[:,Lp-1]=var_rho[:,-2]
    return var_rho
    
def v2rho_2d (var_v):
    [M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[1:M-1,:]=0.5*(var_v[0:Mm-1,:]+var_v[1:M-1,:])
    var_rho[0,:]=var_rho[1,:]
    var_rho[Mp-1,:]=var_rho[-2,:]
    return var_rho

def u2rho_3d (var_u):
    [N,Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,:,1:L-1]=0.5*(var_u[:,:,0:Lm-1]+var_u[:,:,1:L-1])
    var_rho[:,:,0]=var_rho[:,:,1]
    var_rho[:,:,Lp-1]=var_rho[:,:,-2]
    return var_rho
    
def v2rho_3d (var_v):
    [N,M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,1:M-1,:]=0.5*(var_v[:,0:Mm-1,:]+var_v[:,1:M-1,:])
    var_rho[:,0,:]=var_rho[:,1,:]
    var_rho[:,Mp-1,:]=var_rho[:,-2,:]
    return var_rho

def add_new_var_to_dataset_2d(ds, var_name, var_values):
    """
    将一个新的变量添加到现有的 xarray Dataset 中。

    参数:
    ds: xarray.Dataset
        要添加变量的原始 Dataset。
    var_name: str
        新变量的名称。
    var_values: array-like
        用于填充新变量的值。可以是一个二维数组，形状应与 eta_rho 和 xi_rho 坐标匹配。

    返回:
    ds: xarray.Dataset
        更新后的 Dataset，包含新的 DataArray。
    """
    # 获取 eta_rho 和 xi_rho 坐标
    eta_rho = ds.coords['eta_rho']
    xi_rho = ds.coords['xi_rho']

    # 创建新的 DataArray
    new_var = xr.DataArray(var_values, 
                           coords={'eta_rho': eta_rho, 'xi_rho': xi_rho},
                           dims=['eta_rho', 'xi_rho'], 
                           name=var_name)

    # 将新的 DataArray 加入到 Dataset
    ds[var_name] = new_var

    return ds

def add_new_var_to_dataset_3d(ds, var_name, var_values):
    """
    将一个新的变量添加到现有的 xarray Dataset 中。

    参数:
    ds: xarray.Dataset
        要添加变量的原始 Dataset。
    var_name: str
        新变量的名称。
    var_values: array-like
        用于填充新变量的值。可以是一个二维数组，形状应与 eta_rho 和 xi_rho 坐标匹配。

    返回:
    ds: xarray.Dataset
        更新后的 Dataset，包含新的 DataArray。
    """
    # 获取 eta_rho 和 xi_rho 坐标
    eta_rho = ds.coords['eta_rho']
    xi_rho = ds.coords['xi_rho']
    time = ds.coords['time']
    # 创建新的 DataArray
    new_var = xr.DataArray(var_values, 
                           coords={'time':time,'eta_rho': eta_rho, 'xi_rho': xi_rho},
                           dims=['time','eta_rho', 'xi_rho'], 
                           name=var_name)

    # 将新的 DataArray 加入到 Dataset
    ds[var_name] = new_var

    return ds

def tridim(var2d, N):
    """
    Put a 2D matrix in 3D (reproduce it N times).
    
    Parameters:
    - var2d: 2D numpy array
    - N: integer, the number of times to replicate the 2D array in the 3rd dimension
    
    Returns:
    - var3d: 3D numpy array where var2d is replicated N times along the 3rd axis
    """
    M, L = var2d.shape
    var3d = np.reshape(var2d, (1, M, L))  # Reshape the 2D array to 3D
    var3d = np.tile(var3d, (N, 1, 1))  # Replicate the array N times along the first axis
    return var3d




def calc_ispec(k, l, E, ndim=2):
    """
    Calculates the azimuthally-averaged spectrum.

    Parameters
    ----------
    E : ndarray
        The two-dimensional or three-dimensional spectrum.
    k : ndarray
        The wavenumber in the x-direction.
    l : ndarray
        The wavenumber in the y-direction.
    ndim : int, optional
        The dimension of the input data (default is 2).

    Returns
    -------
    kr : ndarray
        The radial wavenumber.
    Er : ndarray
        The azimuthally-averaged spectrum.
    """
    
    # Compute the wavenumber step sizes
    dk = np.abs(k[1] - k[0])
    dl = np.abs(l[1] - l[0])
    
    # Create a meshgrid for k and l
    K, L = np.meshgrid(k, l)
    
    # Compute the wavenumber magnitude
    wv = np.sqrt(K**2 + L**2)
    
    # Find the maximum wavenumber
    kmax = max(np.max(k), np.max(l))
    
    # Get the size of E depending on ndim
    if ndim == 3:
        nl, nk, nomg = E.shape
    elif ndim == 2:
        nomg = 1
    else:
        raise ValueError("ndim should be 2 or 3")
    
    # Compute the radial bin width
    dkr = np.sqrt(dk**2 + dl**2)
    
    # Create the radial wavenumber range
    kr = np.arange(dkr / 2, kmax + dkr / 2, dkr)
    
    # Initialize the output
    Er = np.zeros((len(kr), nomg))
    
    # Loop through each radial bin
    for i in range(len(kr)):
        # Find the wavenumbers within the current radial bin
        fkr = (wv >= kr[i] - dkr / 2) & (wv <= kr[i] + dkr / 2)
        
        # Compute the angular bin width
        dth = np.pi / (np.sum(fkr) - 1)
        
        # Perform the azimuthal averaging
        if ndim == 2:
            Er[i] = np.sum(E[fkr] * (wv[fkr] * dth))
        elif ndim == 3:
            Er[i] = np.sum(np.sum(E[fkr] * (wv[fkr] * dth), axis=0), axis=0)
    
    # Squeeze Er to remove singleton dimensions
    Er = np.squeeze(Er)
    
    return kr, Er




def detrend_demean(un,vn):
     # Removes trends and mean
    un = un - np.mean(un)
    vn = vn - np.mean(vn)
    un = sig.detrend(un, axis=0, type='linear')
    un = sig.detrend(un, axis=1, type='linear')
    vn = sig.detrend(vn, axis=0, type='linear')
    vn = sig.detrend(vn, axis=1, type='linear')
    return un,vn
    
#### new test

def test2_calc_SF2(u, v, N):
    # Create a meshgrid
    X, Y = np.meshgrid(np.arange(-N/2, N/2), np.arange(-N/2, N/2))
    R = np.sqrt(X**2 + Y**2)
    cost = X / R
    sint = Y / R

    # Handle division by zero (where R == 0)
    cost[R == 0] = 0
    sint[R == 0] = 0

    # Compute 2D Fourier transforms
    uh = np.fft.fft2(u)
    vh = np.fft.fft2(v)

    # Compute autocorrelations and cross-correlations
    Cuu = np.fft.fftshift(np.fft.ifft2(uh * np.conj(uh) / (N**2)))
    Cvv = np.fft.fftshift(np.fft.ifft2(vh * np.conj(vh) / (N**2)))
    Cuv = np.fft.fftshift(np.fft.ifft2(uh * np.conj(vh) / (N**2)))
    Cvu = np.fft.fftshift(np.fft.ifft2(vh * np.conj(uh) / (N**2)))

    # # Compute Ruu, Rvv, Ruv, Rvu
    # Ruu = Cuu * cost**2 + Cvv * sint**2 + sint * cost * (Cuv + Cvu)
    # Rvv = Cvv * sint**2 + Cvv * cost**2 + sint * cost * (Cuv + Cvu)
    # Ruv = (Cuu - Cvv) * cost * sint + Cvu * sint**2 - Cuv * cost**2
    # Rvu = (Cuu - Cvv) * cost * sint - Cvu * cost**2 + Cuv * sint**2

    # Compute mean values
    u2 = np.mean(u**2)
    v2 = np.mean(v**2)
    uv = np.mean(u) * np.mean(v)

    # Compute S_transverse and S_longitudinal
    S_transverse = 2 * (u2 * sint**2 + v2 * cost**2 - 2 * uv * sint * cost) - \
                   2 * (Cuu * sint**2 - (Cuv + Cvu) * sint * cost + Cvv * cost**2)

    S_longitudinal = 2 * (u2 * cost**2 + v2 * sint**2 + 2 * uv * sint * cost) - \
                     2 * (Cuu * cost**2 + (Cuv + Cvu) * sint * cost + Cvv * sint**2)

    # Compute radial distances
    rx = np.arange(-N/2, N/2)
    ry = np.arange(-N/2, N/2)
    rr = np.unique(np.sqrt(rx**2 + ry**2))
    dr = np.sqrt(abs(rx[1] - rx[0])**2 + abs(ry[1] - ry[0])**2)

    # Initialize arrays for St_azimuth and Sl_azimuth
    St_azimuth = np.zeros_like(rr)
    Sl_azimuth = np.zeros_like(rr)

    # Compute St_azimuth and Sl_azimuth
    for ii in range(len(rr)):
        pos = (R >= rr[ii] - dr/2) & (R < rr[ii] + dr/2)
        St_azimuth[ii] = np.mean(S_transverse[pos])
        Sl_azimuth[ii] = np.mean(S_longitudinal[pos])

    # Compute S2
    S2 = 0.5 * (St_azimuth + Sl_azimuth)

    return rr, St_azimuth, Sl_azimuth, S2

def test3_calc_SF2(u, v, N):
    # Create a meshgrid
    X, Y = np.meshgrid(np.arange(-N, N), np.arange(-N, N),indexing='ij')
    R = np.sqrt(X**2 + Y**2)
    cost = X / R
    sint = Y / R

    # Handle division by zero (where R == 0)
    cost[R == 0] = 0
    sint[R == 0] = 0

    # Compute 2D Fourier transforms
    uh = np.fft.fft2(u,s=(2*N-1, 2*N-1))
    vh = np.fft.fft2(v,s=(2*N-1, 2*N-1))

    # Compute autocorrelations and cross-correlations
    Cuu = -np.fft.fftshift(np.fft.ifft2(uh * np.conj(uh) / (N**2)))
    Cvv = -np.fft.fftshift(np.fft.ifft2(vh * np.conj(vh) / (N**2)))
    Cuv = -np.fft.fftshift(np.fft.ifft2(uh * np.conj(vh) / (N**2)))
    Cvu = -np.fft.fftshift(np.fft.ifft2(vh * np.conj(uh) / (N**2)))

    # # Compute Ruu, Rvv, Ruv, Rvu
    # Ruu = Cuu * cost**2 + Cvv * sint**2 + sint * cost * (Cuv + Cvu)
    # Rvv = Cvv * sint**2 + Cvv * cost**2 + sint * cost * (Cuv + Cvu)
    # Ruv = (Cuu - Cvv) * cost * sint + Cvu * sint**2 - Cuv * cost**2
    # Rvu = (Cuu - Cvv) * cost * sint - Cvu * cost**2 + Cuv * sint**2

    # Compute mean values
    u2 = np.mean(u**2)
    v2 = np.mean(v**2)
    uv = np.mean(u) * np.mean(v)

    # Compute S_transverse and S_longitudinal
    S_transverse = 2 * (u2 * sint**2 + v2 * cost**2 - 2 * uv * sint * cost) - \
                   2 * (Cuu * sint**2 - (Cuv + Cvu) * sint * cost + Cvv * cost**2)

    S_longitudinal = 2 * (u2 * cost**2 + v2 * sint**2 + 2 * uv * sint * cost) - \
                     2 * (Cuu * cost**2 + (Cuv + Cvu) * sint * cost + Cvv * sint**2)

    # Compute radial distances
    xx = X[N-1:, N-1:]
    yy = Y[N-1:, N-1:]

    # 处理NaN值
    S_transverse = np.nan_to_num(S_transverse)
    S_longitudinal = np.nan_to_num(S_longitudinal)
    
    # 计算各向同性谱
    r, S2L = calc_ispec(xx[0,:], yy[:,0], S_longitudinal[N-1:, N-1:])
    _, S2T = calc_ispec(xx[0,:], yy[:,0], S_transverse[N-1:, N-1:])
    S2=S2L+S2T
    return r, S2L, S2T, S2

def calc_SF3_iso(u, v, N):
    # 生成坐标网格
    x = np.arange(-N+1, N)
    y = np.arange(-N+1, N)
    X, Y = np.meshgrid(x, y, indexing='xy')
    
    R = np.sqrt(X**2 + Y**2)
    # 避免除以零
    R[R == 0] = np.finfo(float).eps
    cost = X / R
    sint = Y / R
    
    # 计算二次项
    u2 = u**2
    v2 = v**2
    uv = u * v
    
    # 二维FFT（自动补零到2N-1）
    uh = np.fft.fft2(u, s=(2*N-1, 2*N-1))
    vh = np.fft.fft2(v, s=(2*N-1, 2*N-1))
    uuh = np.fft.fft2(u2, s=(2*N-1, 2*N-1))
    vvh = np.fft.fft2(v2, s=(2*N-1, 2*N-1))
    uvh = np.fft.fft2(uv, s=(2*N-1, 2*N-1))
    
    # 计算相关函数（注意FFT归一化）
    scale = (2*N-1)**2  # FFT补零后的尺寸
    Cu_uu = np.fft.fftshift(np.fft.ifft2(uh * np.conj(uuh) / scale))
    Cuu_u = np.fft.fftshift(np.fft.ifft2(uuh * np.conj(uh) / scale))
    Cv_vv = np.fft.fftshift(np.fft.ifft2(vh * np.conj(vvh) / scale))
    Cvv_v = np.fft.fftshift(np.fft.ifft2(vvh * np.conj(vh) / scale))
    
    Cuv_u = np.fft.fftshift(np.fft.ifft2(uvh * np.conj(uh) / scale))
    Cuv_v = np.fft.fftshift(np.fft.ifft2(uvh * np.conj(vh) / scale))
    Cu_uv = np.fft.fftshift(np.fft.ifft2(uh * np.conj(uvh) / scale))
    Cv_uv = np.fft.fftshift(np.fft.ifft2(vh * np.conj(uvh) / scale))
    
    Cuu_v = np.fft.fftshift(np.fft.ifft2(uuh * np.conj(vh) / scale))
    Cvv_u = np.fft.fftshift(np.fft.ifft2(vvh * np.conj(uh) / scale))
    Cv_uu = np.fft.fftshift(np.fft.ifft2(vh * np.conj(uuh) / scale))
    Cu_vv = np.fft.fftshift(np.fft.ifft2(uh * np.conj(vvh) / scale))
    
    # 计算S3分量
    S3L_Roy = (cost**3 * (-3*Cu_uu + 3*Cuu_u) +
               cost**2 * sint * (-6*Cu_uv + 3*Cuu_v + 6*Cuv_u - 3*Cv_uu) +
               cost * sint**2 * (-3*Cu_vv + 6*Cuv_v - 6*Cv_uv + 3*Cvv_u) +
               sint**3 * (-3*Cv_vv + 3*Cvv_v))
    
    S3T_Roy = (cost**3 * (-Cu_vv + 2*Cuv_v - 2*Cv_uv + Cvv_u) +
               cost**2 * sint * (4*Cu_uv - 2*Cuu_v - 4*Cuv_u + 2*Cv_uu - 3*Cv_vv + 3*Cvv_v) +
               cost * sint**2 * (-3*Cu_uu + 2*Cu_vv + 3*Cuu_u - 4*Cuv_v + 4*Cv_uv - 2*Cvv_u) +
               sint**3 * (-2*Cu_uv + Cuu_v + 2*Cuv_u - Cv_uu))
    
    # 提取右上半平面
    S3L_Roy1 = -S3L_Roy[N-1:, N-1:] # cause corelation?
    S3T_Roy1 = -S3T_Roy[N-1:, N-1:]
    
    # 生成对应的坐标
    xx = X[N-1:, N-1:]
    yy = Y[N-1:, N-1:]
    
    # 处理NaN值
    S3L_Roy1 = np.nan_to_num(S3L_Roy1)
    S3T_Roy1 = np.nan_to_num(S3T_Roy1)
    
    # 计算各向同性谱
    r, S3L = calc_ispec(xx[0,:], yy[:,0], S3L_Roy1)
    _, S3T = calc_ispec(xx[0,:], yy[:,0], S3T_Roy1)
    
    return r, S3L, S3T  #

def calc_SF3_iso2(u, v, N):
    # 生成坐标网格
    x = np.arange(-N/2, N/2)
    y = np.arange(-N/2, N/2)
    X, Y = np.meshgrid(x, y, indexing='xy')
    
    R = np.sqrt(X**2 + Y**2)
    # 避免除以零
    R[R == 0] = np.finfo(float).eps
    cost = X / R
    sint = Y / R
    
    # 计算二次项
    u2 = u**2
    v2 = v**2
    uv = u * v
    
    # 二
    uh = np.fft.fft2(u)
    vh = np.fft.fft2(v)
    uuh = np.fft.fft2(u2)
    vvh = np.fft.fft2(v2)
    uvh = np.fft.fft2(uv)
    
    # 计算相关函数（注意FFT归一化）
    scale = (N)**2  # FFT补零后的尺寸
    Cu_uu = -np.fft.fftshift(np.fft.ifft2(uh * np.conj(uuh) / scale))
    Cuu_u = -np.fft.fftshift(np.fft.ifft2(uuh * np.conj(uh) / scale))
    Cv_vv = -np.fft.fftshift(np.fft.ifft2(vh * np.conj(vvh) / scale))
    Cvv_v = -np.fft.fftshift(np.fft.ifft2(vvh * np.conj(vh) / scale))
    
    Cuv_u = -np.fft.fftshift(np.fft.ifft2(uvh * np.conj(uh) / scale))
    Cuv_v = -np.fft.fftshift(np.fft.ifft2(uvh * np.conj(vh) / scale))
    Cu_uv = -np.fft.fftshift(np.fft.ifft2(uh * np.conj(uvh) / scale))
    Cv_uv = -np.fft.fftshift(np.fft.ifft2(vh * np.conj(uvh) / scale))
    
    Cuu_v = -np.fft.fftshift(np.fft.ifft2(uuh * np.conj(vh) / scale))
    Cvv_u = -np.fft.fftshift(np.fft.ifft2(vvh * np.conj(uh) / scale))
    Cv_uu = -np.fft.fftshift(np.fft.ifft2(vh * np.conj(uuh) / scale))
    Cu_vv = -np.fft.fftshift(np.fft.ifft2(uh * np.conj(vvh) / scale))
    
    # 计算S3分量
    S3L_2d = (cost**3 * (-3*Cu_uu + 3*Cuu_u) +
               cost**2 * sint * (-6*Cu_uv + 3*Cuu_v + 6*Cuv_u - 3*Cv_uu) +
               cost * sint**2 * (-3*Cu_vv + 6*Cuv_v - 6*Cv_uv + 3*Cvv_u) +
               sint**3 * (-3*Cv_vv + 3*Cvv_v))
    
    S3T_2d = (cost**3 * (-Cu_vv + 2*Cuv_v - 2*Cv_uv + Cvv_u) +
               cost**2 * sint * (4*Cu_uv - 2*Cuu_v - 4*Cuv_u + 2*Cv_uu - 3*Cv_vv + 3*Cvv_v) +
               cost * sint**2 * (-3*Cu_uu + 2*Cu_vv + 3*Cuu_u - 4*Cuv_v + 4*Cv_uv - 2*Cvv_u) +
               sint**3 * (-2*Cu_uv + Cuu_v + 2*Cuv_u - Cv_uu))
    
    # 提取正频
    # print(S3L_2d.shape)
    S3L_2d1 = S3L_2d[int(N/2):, int(N/2):] # 
    S3T_2d1 = S3T_2d[int(N/2):, int(N/2):]
    
    # 生成对应的坐标
    xx = X[int(N/2):, int(N/2):]
    yy = Y[int(N/2):, int(N/2):]
    
    # 处理NaN值
    S3L_2d1 = np.nan_to_num(S3L_2d1)
    S3T_2d1 = np.nan_to_num(S3T_2d1)
    
    # 计算各向同性谱
    r, S3L = calc_ispec2(xx[0,:], yy[:,0], S3L_2d1)
    _, S3T = calc_ispec2(xx[0,:], yy[:,0], S3T_2d1)
    
    return r, S3L, S3T  #

def calc_ispec(k, l, E, ndim=2):
    """计算方位平均谱"""
    dk = abs(k[1] - k[0])
    dl = abs(l[1] - l[0])
    
    # 生成波数网格
    K, L = np.meshgrid(k, l, indexing='xy')
    wv = np.sqrt(K**2 + L**2)
    
    # 确定最大波数
    kmax = np.max([np.max(k), np.max(l)])
    
    # 分箱设置
    dkr = np.sqrt(dk**2 + dl**2)
    kr = np.arange(dkr/2, kmax + dkr/2, dkr)
    
    # 初始化输出
    if ndim == 3:
        nomg = E.shape[2]
    else:
        nomg = 1
    Er = np.zeros((len(kr), nomg), dtype=np.complex128)
    
    for i in range(len(kr)):
        # 选择当前波数区间
        mask = (wv >= kr[i] - dkr/2) & (wv <= kr[i] + dkr/2)
        # 计算角度积分
        if np.sum(mask) > 0:
            dtheta = np.pi / (np.sum(mask) - 1)
            if ndim == 2:
                Er[i] = np.sum(E[mask] * wv[mask] * dtheta)
            else:
                Er[i] = np.sum(E[mask, :] * wv[mask, None] * dtheta, axis=0)
    
    return kr, np.squeeze(Er)  # 取实数部分

def test4_calc_SF2(u, v, N, dx):
    """
    Calculate azimuthally averaged structure functions from velocity fields
    
    Parameters:
    u, v : 2D numpy arrays
        Velocity fields (N x N)
    N : int
        Grid size
    dx : float
        Grid spacing (physical units, e.g., meters)
        
    Returns:
    R_bins : 1D array
        Distance axis (physical units)
    St_azimuth : 1D array  
        Azimuthally averaged transverse structure function
    Sl_azimuth : 1D array
        Azimuthally averaged longitudinal structure function 
    S2 : 1D array
        Combined structure function
    """
    import numpy as np
    from scipy.fft import fft2, ifft2, fftshift
    
    # Create grid
    x = np.arange(-N//2, N//2)
    y = np.arange(-N//2, N//2)
    X, Y = np.meshgrid(x, y, indexing='ij')
    R = np.sqrt(X**2 + Y**2) * dx  # Physical distance matrix
    
    # Handle division by zero for cost/sint
    with np.errstate(divide='ignore', invalid='ignore'):
        cost = X / (R/dx)
        sint = Y / (R/dx)
        cost[np.isnan(cost)] = 0
        sint[np.isnan(sint)] = 0
    
    # Compute velocity correlations
    uh = fft2(u)
    vh = fft2(v)
    Cuu = fftshift(ifft2(uh * np.conj(uh))) / N**2
    Cvv = fftshift(ifft2(vh * np.conj(vh))) / N**2
    Cuv = fftshift(ifft2(uh * np.conj(vh))) / N**2
    Cvu = fftshift(ifft2(vh * np.conj(uh))) / N**2
    
    # Structure functions
    u2 = np.mean(u**2)
    v2 = np.mean(v**2)
    uv = np.mean(u) * np.mean(v)
    
    S_transverse = (2*(u2*sint**2 + v2*cost**2 - 2*uv*sint*cost) - 
                    2*(Cuu*sint**2 - (Cuv+Cvu)*sint*cost + Cvv*cost**2))
    
    S_longitudinal = (2*(u2*cost**2 + v2*sint**2 + 2*uv*sint*cost) - 
                      2*(Cuu*cost**2 + (Cuv+Cvu)*sint*cost + Cvv*sint**2))
    
    # Azimuthal averaging
    R_max = np.max(R)
    R_bins = np.arange(0, np.floor(R_max/dx)+1) * dx
    
    R_discrete = np.round(R / dx).astype(int)
    R_discrete[R_discrete >= len(R_bins)] = len(R_bins)-1
    
    # Initialize output arrays
    Sl_azimuth = np.zeros(len(R_bins))
    St_azimuth = np.zeros(len(R_bins))
    counts = np.zeros(len(R_bins))
    
    # Manual accumulation (equivalent to MATLAB's accumarray)
    for i in range(N):
        for j in range(N):
            idx = R_discrete[i,j]
            Sl_azimuth[idx] += S_longitudinal[i,j]
            St_azimuth[idx] += S_transverse[i,j]
            counts[idx] += 1
    
    # Normalize
    counts[counts == 0] = 1  # Avoid division by zero
    Sl_azimuth /= counts
    St_azimuth /= counts
    
    # Combined structure function
    S2 = 0.5 * (St_azimuth + Sl_azimuth)
    
    return R_bins, St_azimuth, Sl_azimuth, S2


def test4_calc_SF3(u, v, N, dx):
    """
    Calculate azimuthally averaged structure functions from velocity fields
    
    Parameters:
    u, v : 2D numpy arrays
        Velocity fields (N x N)
    N : int
        Grid size
    dx : float
        Grid spacing (physical units, e.g., meters)
        
    Returns:
    R_bins : 1D array
        Distance axis (physical units)
    St_azimuth : 1D array  
        Azimuthally averaged transverse structure function
    Sl_azimuth : 1D array
        Azimuthally averaged longitudinal structure function 
    S2 : 1D array
        Combined structure function
    """
    import numpy as np
    from scipy.fft import fft2, ifft2, fftshift
    
    # Create grid
    x = np.arange(-N//2, N//2)
    y = np.arange(-N//2, N//2)
    X, Y = np.meshgrid(x, y, indexing='ij')
    R = np.sqrt(X**2 + Y**2) * dx  # Physical distance matrix
    
    # Handle division by zero for cost/sint
    with np.errstate(divide='ignore', invalid='ignore'):
        cost = X / (R/dx)
        sint = Y / (R/dx)
        cost[np.isnan(cost)] = 0
        sint[np.isnan(sint)] = 0
    
    # Compute velocity correlations
    # uh = fft2(u)
    # vh = fft2(v)
    # Cuu = fftshift(ifft2(uh * np.conj(uh))) / N**2
    # Cvv = fftshift(ifft2(vh * np.conj(vh))) / N**2
    # Cuv = fftshift(ifft2(uh * np.conj(vh))) / N**2
    # Cvu = fftshift(ifft2(vh * np.conj(uh))) / N**2
    
    # # Structure functions
    # u2 = np.mean(u**2)
    # v2 = np.mean(v**2)
    # uv = np.mean(u) * np.mean(v)
    
    # S_transverse = (2*(u2*sint**2 + v2*cost**2 - 2*uv*sint*cost) - 
    #                 2*(Cuu*sint**2 - (Cuv+Cvu)*sint*cost + Cvv*cost**2))
    
    # S_longitudinal = (2*(u2*cost**2 + v2*sint**2 + 2*uv*sint*cost) - 
    #                   2*(Cuu*cost**2 + (Cuv+Cvu)*sint*cost + Cvv*sint**2))
    u2 = u**2
    v2 = v**2
    uv = u * v
    
    # 二
    uh = fft2(u)
    vh = fft2(v)
    uuh = fft2(u2)
    vvh = fft2(v2)
    uvh = fft2(uv)
    
    # 计算相关函数（注意FFT归一化）
    scale = (N)**2  # FFT补零后的尺寸
    Cu_uu = -fftshift(ifft2(uh * np.conj(uuh) / scale))
    Cuu_u = -fftshift(ifft2(uuh * np.conj(uh) / scale))
    Cv_vv = -fftshift(ifft2(vh * np.conj(vvh) / scale))
    Cvv_v = -fftshift(ifft2(vvh * np.conj(vh) / scale))
    
    Cuv_u = -fftshift(ifft2(uvh * np.conj(uh) / scale))
    Cuv_v = -fftshift(ifft2(uvh * np.conj(vh) / scale))
    Cu_uv = -fftshift(ifft2(uh * np.conj(uvh) / scale))
    Cv_uv = -fftshift(ifft2(vh * np.conj(uvh) / scale))
    
    Cuu_v = -fftshift(ifft2(uuh * np.conj(vh) / scale))
    Cvv_u = -fftshift(ifft2(vvh * np.conj(uh) / scale))
    Cv_uu = -fftshift(ifft2(vh * np.conj(uuh) / scale))
    Cu_vv = -fftshift(ifft2(uh * np.conj(vvh) / scale))

    S3L_2d = (cost**3 * (-3*Cu_uu + 3*Cuu_u) +
               cost**2 * sint * (-6*Cu_uv + 3*Cuu_v + 6*Cuv_u - 3*Cv_uu) +
               cost * sint**2 * (-3*Cu_vv + 6*Cuv_v - 6*Cv_uv + 3*Cvv_u) +
               sint**3 * (-3*Cv_vv + 3*Cvv_v))
    
    S3T_2d = (cost**3 * (-Cu_vv + 2*Cuv_v - 2*Cv_uv + Cvv_u) +
               cost**2 * sint * (4*Cu_uv - 2*Cuu_v - 4*Cuv_u + 2*Cv_uu - 3*Cv_vv + 3*Cvv_v) +
               cost * sint**2 * (-3*Cu_uu + 2*Cu_vv + 3*Cuu_u - 4*Cuv_v + 4*Cv_uv - 2*Cvv_u) +
               sint**3 * (-2*Cu_uv + Cuu_v + 2*Cuv_u - Cv_uu))
    
    # Azimuthal averaging
    R_max = np.max(R)
    R_bins = np.arange(0, np.floor(R_max/dx)+1) * dx
    
    R_discrete = np.round(R / dx).astype(int)
    R_discrete[R_discrete >= len(R_bins)] = len(R_bins)-1
    
    # Initialize output arrays
    Sl_azimuth = np.zeros(len(R_bins))
    St_azimuth = np.zeros(len(R_bins))
    counts = np.zeros(len(R_bins))
    
    # Manual accumulation (equivalent to MATLAB's accumarray)
    for i in range(N):
        for j in range(N):
            idx = R_discrete[i,j]
            Sl_azimuth[idx] +=  S3L_2d[i,j]
            St_azimuth[idx] +=  S3T_2d[i,j]
            counts[idx] += 1
    
    # Normalize
    counts[counts == 0] = 1  # Avoid division by zero
    Sl_azimuth /= counts
    St_azimuth /= counts
    
    # Combined structure function
    # S3 =  (St_azimuth + Sl_azimuth)
    
    return R_bins, St_azimuth, Sl_azimuth

def test4_calc_SF3_positive_freq(u, v, N, dx):
    """
    Calculate azimuthally averaged structure functions (positive frequencies only)
    
    Parameters:
    u, v : 2D numpy arrays
        Velocity fields (N x N)
    N : int
        Grid size
    dx : float
        Grid spacing (physical units, e.g., meters)
        
    Returns:
    R_bins : 1D array
        Distance axis (physical units)
    St_azimuth : 1D array  
        Azimuthally averaged transverse structure function (positive freq)
    Sl_azimuth : 1D array
        Azimuthally averaged longitudinal structure function (positive freq)
    """
    import numpy as np
    from scipy.fft import fft2, ifft2, fftshift
    
    # Create grid for positive frequencies only
    x = np.arange(0, N//2)  # Only positive x frequencies
    y = np.arange(0, N//2)  # Only positive y frequencies
    X, Y = np.meshgrid(x, y, indexing='ij')
    R = np.sqrt(X**2 + Y**2) * dx  # Physical distance matrix
    
    # Handle division by zero for cost/sint
    with np.errstate(divide='ignore', invalid='ignore'):
        cost = X / (R/dx + 1e-12)
        sint = Y / (R/dx + 1e-12)
        cost[np.isnan(cost)] = 0
        sint[np.isnan(sint)] = 0
    
    # Compute velocity correlations
    u2 = u**2
    v2 = v**2
    uv = u * v
    
    # Compute FFTs - only need positive frequencies
    uh = fft2(u)[:N//2, :N//2]
    vh = fft2(v)[:N//2, :N//2]
    uuh = fft2(u2)[:N//2, :N//2]
    vvh = fft2(v2)[:N//2, :N//2]
    uvh = fft2(uv)[:N//2, :N//2]
    
    scale = N**2
    
    # Compute correlation functions for positive frequencies only
    Cu_uu = -fftshift(ifft2(uh * np.conj(uuh) / scale))
    Cuu_u = -fftshift(ifft2(uuh * np.conj(uh) / scale))
    Cv_vv = -fftshift(ifft2(vh * np.conj(vvh) / scale))
    Cvv_v = -fftshift(ifft2(vvh * np.conj(vh) / scale))
    
    Cuv_u = -fftshift(ifft2(uvh * np.conj(uh) / scale))
    Cuv_v = -fftshift(ifft2(uvh * np.conj(vh) / scale))
    Cu_uv = -fftshift(ifft2(uh * np.conj(uvh) / scale))
    Cv_uv = -fftshift(ifft2(vh * np.conj(uvh) / scale))
    
    Cuu_v = -fftshift(ifft2(uuh * np.conj(vh) / scale))
    Cvv_u = -fftshift(ifft2(vvh * np.conj(uh) / scale))
    Cv_uu = -fftshift(ifft2(vh * np.conj(uuh) / scale))
    Cu_vv = -fftshift(ifft2(uh * np.conj(vvh) / scale))

    # Calculate structure functions for positive frequencies
    S3L_2d = (cost**3 * (-3*Cu_uu + 3*Cuu_u) +
              cost**2 * sint * (-6*Cu_uv + 3*Cuu_v + 6*Cuv_u - 3*Cv_uu) +
              cost * sint**2 * (-3*Cu_vv + 6*Cuv_v - 6*Cv_uv + 3*Cvv_u) +
              sint**3 * (-3*Cv_vv + 3*Cvv_v))
    
    S3T_2d = (cost**3 * (-Cu_vv + 2*Cuv_v - 2*Cv_uv + Cvv_u) +
              cost**2 * sint * (4*Cu_uv - 2*Cuu_v - 4*Cuv_u + 2*Cv_uu - 3*Cv_vv + 3*Cvv_v) +
              cost * sint**2 * (-3*Cu_uu + 2*Cu_vv + 3*Cuu_u - 4*Cuv_v + 4*Cv_uv - 2*Cvv_u) +
              sint**3 * (-2*Cu_uv + Cuu_v + 2*Cuv_u - Cv_uu))
    
    # Azimuthal averaging for positive frequencies
    R_max = np.max(R)
    R_bins = np.arange(0, np.floor(R_max/dx)+1) * dx
    
    R_discrete = np.round(R / dx).astype(int)
    R_discrete[R_discrete >= len(R_bins)] = len(R_bins)-1
    
    # Initialize output arrays
    Sl_azimuth = np.zeros(len(R_bins))
    St_azimuth = np.zeros(len(R_bins))
    counts = np.zeros(len(R_bins))
    
    # Manual accumulation for positive frequencies
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            idx = R_discrete[i,j]
            Sl_azimuth[idx] += S3L_2d[i,j].real  # Take real part
            St_azimuth[idx] += S3T_2d[i,j].real
            counts[idx] += 1
    
    # Normalize
    counts[counts == 0] = 1  # Avoid division by zero
    Sl_azimuth /= counts
    St_azimuth /= counts
    
    return R_bins, St_azimuth, Sl_azimuth

In [3]:
outpur_dir='/meddy/simingzhang/Data/HIT2D/2dt2'


grid_dir='/meddy/simingzhang/Data/HIT2D/2dt2'

os.chdir(grid_dir)

import scipy.io as sio
from tqdm import tqdm
grid=sio.loadmat('HIT2D_Parameters.mat')
Diag=grid['Diag']
mm=grid['mm']
kk=grid['kk']
u=np.zeros((199,512,512))
v=np.zeros_like(u)

for i in tqdm(range(0,199)):
    fname=f'HIT2D_t_{i+4000}.mat'
    data=sio.loadmat(fname)
    hq=data['hq']
    
    hpsi=Diag*hq;
    hu1=-1j*mm*hpsi;
    hw1=1j*kk*hpsi;
    u[i,:,:]=np.real(np.fft.ifft2(hu1))
    v[i,:,:]=np.real(np.fft.ifft2(hw1))

pm=np.ones((512,512))
pn=np.ones((512,512))

100%|██████████████████████████████████████████████████████████████████████| 199/199 [00:20<00:00,  9.82it/s]


In [4]:
u[:,np.newaxis,:,:].shape

(199, 1, 512, 512)

In [5]:
u1=0.5*(u[:,np.newaxis,:,1:]+u[:,np.newaxis,:,:-1])
v1=0.5*(v[:,np.newaxis,1:,:]+v[:,np.newaxis,:-1,:])

In [6]:

time_size = 199      # 时间步长（如10天）
depth_size = 1      # 深度层数（如5层）
eta_size = 512      # eta_rho方向网格数
xi_size = 512       # xi_rho方向网格数

# 2. 创建坐标值
time_coords = np.arange(1, time_size + 1)                     
depth_coords = [-2]                                           # 深度坐标（单位：米）
eta_coords = np.arange(1,eta_size+1)                              # eta_rho坐标（索引）
xi_coords = np.arange(1,xi_size+1)                                # xi_rho坐标（索引）
dt = 1e-3  # 时间步长1毫秒
time_sec = time_coords * dt  # 实际时间 = 时间索引 * 时间步长
x_rho,y_rho=np.meshgrid(np.linspace(0, 6.2832*(1-1/512), 512),np.linspace(0, 6.2832*(1-1/512), 512),indexing='ij')
x_rho=x_rho.T
y_rho=y_rho.T
dx=6.2832/512
pm = np.ones((eta_size, xi_size)) / dx
pn = np.ones((eta_size, xi_size)) / dx
# 3. 生成数据（这里用全零占位，实际可用真实数据替换）

ds = xr.Dataset(
    data_vars={
        "u_rho": (["time", "depth", "eta_rho", "xi_rho"], u[:,np.newaxis,:,:]),
        "v_rho": (["time", "depth", "eta_rho", "xi_rho"], v[:,np.newaxis,:,:]),
        "u": (["time", "depth", "eta_u", "xi_u"], u1),
        "v": (["time", "depth", "eta_v", "xi_v"], v1),
        "ocean_time": ("time", time_sec),
        "lon_rho": (["eta_rho", "xi_rho"], x_rho),
        "lat_rho": (["eta_rho", "xi_rho"], y_rho),
        "pm": (["eta_rho", "xi_rho"], pm),
        "pn": (["eta_rho", "xi_rho"], pn),
        "f": (["eta_rho", "xi_rho"], pn/pn),
    },
    coords={
        "time": time_coords,
        "depth": depth_coords,
        "eta_rho": eta_coords,
        "xi_rho": xi_coords,
        "eta_u": eta_coords,
        "xi_u": xi_coords[:-1],
        "eta_v": eta_coords[:-1],
        "xi_v": xi_coords,
    }
)

# 查看结果
print(ds)

<xarray.Dataset> Size: 2GB
Dimensions:     (time: 199, depth: 1, eta_rho: 512, xi_rho: 512, eta_u: 512,
                 xi_u: 511, eta_v: 511, xi_v: 512)
Coordinates:
  * time        (time) int64 2kB 1 2 3 4 5 6 7 8 ... 193 194 195 196 197 198 199
  * depth       (depth) int64 8B -2
  * eta_rho     (eta_rho) int64 4kB 1 2 3 4 5 6 7 ... 507 508 509 510 511 512
  * xi_rho      (xi_rho) int64 4kB 1 2 3 4 5 6 7 ... 506 507 508 509 510 511 512
  * eta_u       (eta_u) int64 4kB 1 2 3 4 5 6 7 ... 506 507 508 509 510 511 512
  * xi_u        (xi_u) int64 4kB 1 2 3 4 5 6 7 8 ... 505 506 507 508 509 510 511
  * eta_v       (eta_v) int64 4kB 1 2 3 4 5 6 7 ... 505 506 507 508 509 510 511
  * xi_v        (xi_v) int64 4kB 1 2 3 4 5 6 7 8 ... 506 507 508 509 510 511 512
Data variables:
    u_rho       (time, depth, eta_rho, xi_rho) float64 417MB 0.4364 ... 8.261
    v_rho       (time, depth, eta_rho, xi_rho) float64 417MB 5.986 ... 7.645
    u           (time, depth, eta_u, xi_u) float64 417MB 0.2938

In [7]:
eta_coords

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [8]:
ds

<xarray.Dataset> Size: 2GB
Dimensions:     (time: 199, depth: 1, eta_rho: 512, xi_rho: 512, eta_u: 512,
                 xi_u: 511, eta_v: 511, xi_v: 512)
Coordinates:
  * time        (time) int64 2kB 1 2 3 4 5 6 7 8 ... 193 194 195 196 197 198 199
  * depth       (depth) int64 8B -2
  * eta_rho     (eta_rho) int64 4kB 1 2 3 4 5 6 7 ... 507 508 509 510 511 512
  * xi_rho      (xi_rho) int64 4kB 1 2 3 4 5 6 7 ... 506 507 508 509 510 511 512
  * eta_u       (eta_u) int64 4kB 1 2 3 4 5 6 7 ... 506 507 508 509 510 511 512
  * xi_u        (xi_u) int64 4kB 1 2 3 4 5 6 7 8 ... 505 506 507 508 509 510 511
  * eta_v       (eta_v) int64 4kB 1 2 3 4 5 6 7 ... 505 506 507 508 509 510 511
  * xi_v        (xi_v) int64 4kB 1 2 3 4 5 6 7 8 ... 506 507 508 509 510 511 512
Data variables:
    u_rho       (time, depth, eta_rho, xi_rho) float64 417MB 0.4364 ... 8.261
    v_rho       (time, depth, eta_rho, xi_rho) float64 417MB 5.986 ... 7.645
    u           (time, depth, eta_u, xi_u) float64 417MB 0.2938 0.3326 ... 8.207
    v           (time, depth, eta_v, xi_v) float64 417MB 6.497 5.406 ... 7.652
    ocean_time  (time) float64 2kB 0.001 0.002 0.003 0.004 ... 0.197 0.198 0.199
    lon_rho     (eta_rho, xi_rho) float64 2MB 0.0 0.01227 ... 6.259 6.271
    lat_rho     (eta_rho, xi_rho) float64 2MB 0.0 0.0 0.0 ... 6.271 6.271 6.271
    pm          (eta_rho, xi_rho) float64 2MB 81.49 81.49 81.49 ... 81.49 81.49
    pn          (eta_rho, xi_rho) float64 2MB 81.49 81.49 81.49 ... 81.49 81.49
    f           (eta_rho, xi_rho) float64 2MB 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0

In [9]:
ds.to_netcdf('/meddy/simingzhang/Data/Parcels_data/HIT2d_new.nc')

# Analysis

In [15]:
np.min(ds['pm'].values)

81.48714031066973

In [16]:
ds['pm'].values[0,:].shape

(512,)

In [17]:
ds['lon_rho'].values[2,:]

array([0.        , 0.01227187, 0.02454375, 0.03681562, 0.0490875 ,
       0.06135938, 0.07363125, 0.08590312, 0.098175  , 0.11044687,
       0.12271875, 0.13499063, 0.1472625 , 0.15953438, 0.17180625,
       0.18407813, 0.19635   , 0.20862187, 0.22089375, 0.23316562,
       0.2454375 , 0.25770938, 0.26998125, 0.28225312, 0.294525  ,
       0.30679687, 0.31906875, 0.33134062, 0.3436125 , 0.35588437,
       0.36815625, 0.38042813, 0.3927    , 0.40497187, 0.41724375,
       0.42951563, 0.4417875 , 0.45405937, 0.46633125, 0.47860313,
       0.490875  , 0.50314687, 0.51541875, 0.52769062, 0.5399625 ,
       0.55223437, 0.56450625, 0.57677813, 0.58905   , 0.60132188,
       0.61359375, 0.62586562, 0.6381375 , 0.65040937, 0.66268125,
       0.67495313, 0.687225  , 0.69949688, 0.71176875, 0.72404062,
       0.7363125 , 0.74858437, 0.76085625, 0.77312812, 0.7854    ,
       0.79767188, 0.80994375, 0.82221563, 0.8344875 , 0.84675937,
       0.85903125, 0.87130312, 0.883575  , 0.89584687, 0.90811

In [18]:
np.max(ds['u'].values)

3438.055908686896

In [86]:
pm.shape

(512, 512)